# Ethiopian Grade 12 Exam Pattern Project (Google Colab)

This notebook converts your scripts into a Colab workflow:
1. Download Grade 12 exam PDFs
2. OCR text extraction from PDFs
3. Build a simple question-pattern analysis baseline


## Notes
- Run cells top to bottom.
- In Colab, OCR dependencies are installed automatically.
- Data is saved under `/content/ethiopian_exam_project`.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr
!pip -q install pandas scikit-learn matplotlib seaborn easyocr

In [ ]:
import os
import re
import csv
import time
import tempfile
import subprocess
import urllib.request
from collections import Counter, defaultdict
import pandas as pd

BASE_URL = "https://www.ethiobookreview.com/assets/exams/"
PROJECT_ROOT = "/content/ethiopian_exam_project"
DOWNLOAD_DIR = os.path.join(PROJECT_ROOT, "downloads")
EXTRACTED_DIR = os.path.join(PROJECT_ROOT, "extracted_text")
OUTPUT_CSV = os.path.join(PROJECT_ROOT, "ethiopian_exams_extracted.csv")
PATTERN_CSV = os.path.join(PROJECT_ROOT, "pattern_summary.csv")

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(EXTRACTED_DIR, exist_ok=True)

NATURAL_SCIENCE_SUBJECTS = ["english","mathematics","physics","chemistry","biology","civics"]
SOCIAL_SCIENCE_SUBJECTS = ["geography","history","economics"]
YEARS = ["2005", "2006", "2007", "2008", "2009", "2010"]
TYPES = ["questions", "answers"]

In [ ]:
def download_file(url, filepath, max_retries=3):
    for attempt in range(max_retries):
        try:
            urllib.request.urlretrieve(url, filepath)
            return True
        except Exception as e:
            print(f"Attempt {attempt+1} failed for {os.path.basename(filepath)}: {e}")
            time.sleep(2)
    return False

def download_subject_year(subject, year, target_dir):
    results = []
    for file_type in TYPES:
        filename = f"{subject}-{year}-{file_type}.pdf"
        url = f"{BASE_URL}{filename}"
        filepath = os.path.join(target_dir, filename)

        if os.path.exists(filepath):
            results.append((filename, "already_downloaded"))
            continue

        ok = download_file(url, filepath)
        results.append((filename, "success" if ok else "failed"))
        time.sleep(0.5)
    return results

def run_downloader():
    all_subjects = {"natural_science": NATURAL_SCIENCE_SUBJECTS, "social_science": SOCIAL_SCIENCE_SUBJECTS}
    summary = defaultdict(lambda: {"downloaded": 0, "failed": 0, "already": 0})

    for stream, subjects in all_subjects.items():
        for subject in subjects:
            subject_dir = os.path.join(DOWNLOAD_DIR, stream, subject)
            os.makedirs(subject_dir, exist_ok=True)
            for year in YEARS:
                for _, status in download_subject_year(subject, year, subject_dir):
                    if status == "success": summary[subject]["downloaded"] += 1
                    elif status == "failed": summary[subject]["failed"] += 1
                    else: summary[subject]["already"] += 1

    return pd.DataFrame(summary).T.sort_index()

download_report = run_downloader()
download_report

In [ ]:
# OCR engine: EasyOCR (primary) with Tesseract fallback for environments
# without torch/GPU. EasyOCR gives noticeably better accuracy on scanned
# exam PDFs than Tesseract.
try:
    import easyocr
    _OCR_BACKEND = 'easyocr'
    _easyocr_reader = easyocr.Reader(['en'], gpu=True)
except Exception as _e:
    print(f'EasyOCR unavailable ({_e}); falling back to Tesseract.')
    _OCR_BACKEND = 'tesseract'
    _easyocr_reader = None

def pdf_to_images(pdf_path, output_dir, dpi=200):
    cmd = ['pdftoppm', '-r', str(dpi), '-png', pdf_path, os.path.join(output_dir, 'page')]
    subprocess.run(cmd, capture_output=True, check=False)
    images = sorted([f for f in os.listdir(output_dir) if f.startswith('page') and f.endswith('.png')])
    return [os.path.join(output_dir, img) for img in images]

def extract_text_from_image(image_path, min_confidence=0.3):
    if _OCR_BACKEND == 'easyocr':
        results = _easyocr_reader.readtext(image_path, detail=1, paragraph=False)
        pieces, confs = [], []
        for _, txt, conf in results:
            if conf >= min_confidence and txt.strip():
                pieces.append(txt)
                confs.append(conf)
        text = ' '.join(pieces)
        mean_conf = sum(confs) / len(confs) if confs else 0.0
        return text, mean_conf
    cmd = ['tesseract', image_path, 'stdout', '-l', 'eng', '--psm', '6']
    result = subprocess.run(cmd, capture_output=True, text=True)
    text = result.stdout if result.returncode == 0 else ''
    return text, (0.6 if text.strip() else 0.0)

def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip() if text else ''

def parse_filename(filename):
    parts = filename.replace('.pdf', '').split('-')
    subject = parts[0] if len(parts) > 0 else 'unknown'
    year = parts[1] if len(parts) > 1 else 'unknown'
    ftype = parts[2] if len(parts) > 2 else 'unknown'
    return subject, year, ftype

def extract_all_pdfs():
    all_records = []
    natural_subjects = set(NATURAL_SCIENCE_SUBJECTS)
    social_subjects = set(SOCIAL_SCIENCE_SUBJECTS)
    temp_dir = tempfile.mkdtemp(prefix='pdf_ocr_')

    pdf_paths = []
    for root, _, files in os.walk(DOWNLOAD_DIR):
        pdf_paths += [os.path.join(root, f) for f in files if f.endswith('.pdf')]

    for i, pdf_path in enumerate(sorted(pdf_paths), 1):
        filename = os.path.basename(pdf_path)
        subject, year, ftype = parse_filename(filename)
        stream = 'natural_science' if subject in natural_subjects else ('social_science' if subject in social_subjects else 'unknown')

        for f in os.listdir(temp_dir):
            fp = os.path.join(temp_dir, f)
            if os.path.isfile(fp):
                os.remove(fp)

        images = pdf_to_images(pdf_path, temp_dir, dpi=200)
        page_texts, page_confs = [], []
        for img in images:
            t, c = extract_text_from_image(img)
            page_texts.append(t)
            page_confs.append(c)
        text_full = clean_text('\n\n'.join(page_texts))
        mean_conf = sum(page_confs) / len(page_confs) if page_confs else 0.0

        subject_dir = os.path.join(EXTRACTED_DIR, stream, subject)
        os.makedirs(subject_dir, exist_ok=True)
        txt_path = os.path.join(subject_dir, filename.replace('.pdf', '.txt'))
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(text_full)

        all_records.append({
            'subject': subject,
            'year': year,
            'type': ftype,
            'stream': stream,
            'filename': filename,
            'text_length': len(text_full),
            'ocr_engine': _OCR_BACKEND,
            'ocr_confidence': round(mean_conf, 3),
            'text_preview': text_full[:500],
            'text_full': text_full
        })
        if i % 10 == 0:
            print(f'Processed {i}/{len(pdf_paths)} files')

    df = pd.DataFrame(all_records)
    df.to_csv(OUTPUT_CSV, index=False)
    return df

exams_df = extract_all_pdfs()
exams_df.head()

## Question extraction & topic taxonomy

Replaces the old keyword-only approach with:
1. **Question extraction** — split each exam's OCR text into individual questions (with options), so trend analysis runs on actual question stems, not the full document (which mixes instructions, page numbers, and OCR noise).
2. **Hybrid TF-IDF taxonomy** — start from hand-curated `TOPIC_RULES` (in `topics.py`) and augment each topic's keyword set with terms learned from the corpus that statistically co-occur with seed keywords. Catches synonyms and related vocabulary that pure keyword matching misses.

In [ ]:
# Use the project modules for question parsing + taxonomy.
# In Colab, make sure the repo files are on sys.path:
import sys
_repo_dir = os.getcwd()
if _repo_dir not in sys.path:
    sys.path.insert(0, _repo_dir)

from question_extractor import extract_questions, extract_question_texts
from topic_classifier import build_taxonomy, score_text
from topics import TOPIC_RULES

def clean_question_text(raw):
    """Lightweight noise removal kept for backward compatibility with cell-12."""
    if not isinstance(raw, str):
        return ''
    t = re.sub(r'[_~`|\\]+', ' ', raw)
    t = re.sub(r'\s+', ' ', t)
    t = re.sub(r'(?<![A-Za-z])[A-Da-d][\)\.:/](?![A-Za-z])', ' ', t)
    t = re.sub(r'(?:(?<=\s)|^)(?:q(?:uestion)?\s*)?\d{1,3}[\)\.:-]', ' ', t, flags=re.I)
    t = re.sub(r'[^A-Za-z0-9%+\-=/\(\)\.,;:?!\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

exams_df['clean_text'] = exams_df['text_full'].fillna('').apply(clean_question_text)

# Pull individual question stems out of each exam (questions papers only).
exams_df['questions'] = exams_df.apply(
    lambda r: extract_question_texts(r['text_full']) if r['type'] == 'questions' else [],
    axis=1
)
exams_df['n_questions'] = exams_df['questions'].apply(len)
exams_df[['filename','n_questions']].head()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

question_df = exams_df[exams_df['type'] == 'questions'].copy()
question_df['year_num'] = pd.to_numeric(question_df['year'], errors='coerce')

# Build a hybrid TF-IDF taxonomy per subject from the actual question corpus.
# Falls back to seed-only keywords if a subject has no extracted questions yet.
taxonomies = {}
for subject in question_df['subject'].dropna().unique():
    if subject not in TOPIC_RULES:
        continue
    corpus = []
    for qs in question_df.loc[question_df['subject']==subject, 'questions']:
        corpus.extend(qs)
    taxonomies[subject] = build_taxonomy(subject, corpus)

# Score each exam against its subject taxonomy, aggregating per question.
records = []
for _, r in question_df.iterrows():
    tax = taxonomies.get(r['subject'])
    if tax is None:
        continue
    per_topic = {topic: 0.0 for topic in tax.topics}
    questions_or_full = r['questions'] or [r['clean_text']]
    for q in questions_or_full:
        for topic, s in score_text(q, tax).items():
            per_topic[topic] += s
    for topic, score in per_topic.items():
        records.append({
            'subject': r['subject'],
            'year_num': r['year_num'],
            'topic': topic,
            'score': round(score, 3),
        })

trend_df = (pd.DataFrame(records)
              .groupby(['subject','year_num','topic'], as_index=False)['score'].sum())

# Per-year trend charts
for subject in sorted(trend_df['subject'].dropna().unique()):
    sub = trend_df[trend_df['subject']==subject]
    if sub.empty: continue
    plt.figure(figsize=(10,4))
    sns.lineplot(data=sub, x='year_num', y='score', hue='topic', marker='o')
    plt.title(f'{subject.title()} topic trend by year (TF-IDF taxonomy)')
    plt.xlabel('Exam Year (E.C.)')
    plt.ylabel('Topic score')
    plt.legend(bbox_to_anchor=(1.02,1), loc='upper left')
    plt.tight_layout()
    plt.show()

# Next-exam likely topics: weighted-average baseline (replaced in Phase 3).
pred_rows = []
for (subject, topic), g in trend_df.groupby(['subject','topic']):
    g = g.sort_values('year_num').dropna(subset=['year_num'])
    if g.empty: continue
    y = g['score'].values.astype(float)
    x = g['year_num'].values.astype(float)
    recent = y[-3:] if len(y) >= 3 else y
    recent_mean = float(np.mean(recent))
    latest = float(y[-1])
    slope = float(np.polyfit(x, y, 1)[0]) if len(y) >= 2 else 0.0
    likely = 0.5*recent_mean + 0.3*slope + 0.2*latest
    pred_rows.append({
        'subject': subject, 'topic': topic,
        'recent_mean': recent_mean, 'trend_slope': slope, 'latest': latest,
        'likely_score': likely,
    })

pred_df = pd.DataFrame(pred_rows).sort_values(['subject','likely_score'], ascending=[True, False])
next_topics = pred_df.groupby('subject').head(5)
next_topics


## Next steps for your AI model
- Use `pattern_summary.csv` to identify repeated concepts by subject.
- Train a text classifier on `text_full` with labels like subject/year/type.
- Extract question blocks (split by numbering) and train a topic model to detect recurring question forms.
- Build a study guide from top recurring keywords + question templates.

## Use textbook links in the workflow
This cell downloads the linked Grade 11/12 textbooks and builds a simple topic-reference table to align exam topics with textbook coverage.

In [ ]:
TEXTBOOK_DIR = os.path.join(PROJECT_ROOT, 'textbooks')
os.makedirs(TEXTBOOK_DIR, exist_ok=True)

TEXTBOOK_LINKS = [
    ('g12','biology','https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_BIOLOGY_CURRICULUM_Language_ENGLISH_Retrieved_20150101.pdf'),
    ('g12','civics','https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_CIVICS_Language_ENGLISH_Retrieved_20150101.pdf'),
    ('g12','chemistry','https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_CHEMISTRY_Language_ENGLISH_Retrieved_20150101.pdf'),
    ('g12','english','https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_ENGLISH_Language_ENGLISH_Retrieved_20200601.pdf'),
    ('g12','mathematics','https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_MATH_Language_ENGLISH_Retrieved_20200601.pdf'),
    ('g12','physics','https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_PHYSICS_Language_ENGLISH_Retrieved_20200601.pdf'),
    ('g11','physics','https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_PHYSICS_Language_ENGLISH_Retrieved_20200601.pdf'),
    ('g11','mathematics','https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_MATH_Language_ENGLISH_Retrieved_20200601.pdf'),
    ('g11','english','https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_ENGLISH_Language_ENGLISH_Retrieved_20150101.pdf'),
    ('g11','civics','https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_CIVICS_Language_ENGLISH_Retrieved_20150101.pdf'),
    ('g11','chemistry','https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_CHEMISTRY_Language_ENGLISH_Retrieved_20150101.pdf'),
    ('g11','biology','https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_BIOLOGY_CURRICULUM_Language_ENGLISH_To_Grade_12_Retrieved_20150101.pdf'),
]

def download_textbooks(links):
    rows=[]
    for grade, subject, url in links:
        fname=f'{grade}_{subject}.pdf'
        fpath=os.path.join(TEXTBOOK_DIR,fname)
        if not os.path.exists(fpath):
            ok=download_file(url,fpath,max_retries=2)
            status='downloaded' if ok else 'failed'
        else:
            status='already_exists'
        rows.append({'grade':grade,'subject':subject,'file':fpath,'status':status})
    return pd.DataFrame(rows)

textbook_download_report = download_textbooks(TEXTBOOK_LINKS)
textbook_download_report


In [ ]:
# OCR textbooks and map topic coverage so exam trends can be aligned to chapters/concepts.
# Uses the same TF-IDF taxonomy as the exam scoring, so textbook and exam scores
# are directly comparable.

def extract_textbook_text(pdf_path):
    tmp = tempfile.mkdtemp(prefix='tb_ocr_')
    imgs = pdf_to_images(pdf_path, tmp, dpi=170)
    parts = []
    for img in imgs:
        t, _ = extract_text_from_image(img)
        parts.append(t)
    return clean_question_text(' '.join(parts))

tb_rows = []
for _, r in textbook_download_report.iterrows():
    if r['status'] == 'failed':
        continue
    tax = taxonomies.get(r['subject'])
    if tax is None:
        continue
    tb_text = extract_textbook_text(r['file'])
    for topic, s in score_text(tb_text, tax).items():
        tb_rows.append({
            'grade': r['grade'],
            'subject': r['subject'],
            'topic': topic,
            'textbook_score': s,
        })

textbook_topic_df = pd.DataFrame(tb_rows)
aligned = (next_topics.merge(textbook_topic_df, on=['subject','topic'], how='left')
                       .fillna({'textbook_score': 0})
                       .sort_values(['subject','likely_score'], ascending=[True, False]))
aligned.head(20)


## Grade 11 & 12 textbook links (Ethiopia Learning)
Use these as source materials for topic mapping and revision datasets:

### Grade 12
- Biology: https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_BIOLOGY_CURRICULUM_Language_ENGLISH_Retrieved_20150101.pdf
- Civics: https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_CIVICS_Language_ENGLISH_Retrieved_20150101.pdf
- Chemistry: https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_CHEMISTRY_Language_ENGLISH_Retrieved_20150101.pdf
- English: https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_ENGLISH_Language_ENGLISH_Retrieved_20200601.pdf
- Math: https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_MATH_Language_ENGLISH_Retrieved_20200601.pdf
- Physics: https://files.ethiopialearning.com/textbooks/Grade%2012/Grade_12_Subject_PHYSICS_Language_ENGLISH_Retrieved_20200601.pdf

### Grade 11
- Physics: https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_PHYSICS_Language_ENGLISH_Retrieved_20200601.pdf
- Math: https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_MATH_Language_ENGLISH_Retrieved_20200601.pdf
- English: https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_ENGLISH_Language_ENGLISH_Retrieved_20150101.pdf
- Civics: https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_CIVICS_Language_ENGLISH_Retrieved_20150101.pdf
- Chemistry: https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_CHEMISTRY_Language_ENGLISH_Retrieved_20150101.pdf
- Biology: https://files.ethiopialearning.com/textbooks/Grade%2011/Grade_11_Subject_BIOLOGY_CURRICULUM_Language_ENGLISH_To_Grade_12_Retrieved_20150101.pdf
